# Lab 08-02 · Boolean Algebra and Logic Gates

Companion to chapter [08-02](../../chapters/08-02-boolean-algebra-and-gates.md). Handout: [README.md](README.md).

A Boolean function of $n$ inputs is completely described by its truth table, and a truth table of $2^n$ rows fits in one integer. Everything below works on those integers, so every claim is checked on *every* row. Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import itertools
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def rows(n):
    # all input rows, row i = binary of i with the first variable most significant
    return [tuple((i >> (n - 1 - k)) & 1 for k in range(n)) for i in range(2 ** n)]


def lut(f, n):
    # the truth table as an integer: bit i is f(row i)
    return sum(int(bool(f(*r))) << i for i, r in enumerate(rows(n)))


def same(f, g, n):
    return lut(f, n) == lut(g, n)


NOT = lambda x: 1 - x

## Part 1 · Perfect induction

Each theorem of §3.3 is a pair of functions that must agree on every row. Its dual swaps AND with OR (and 0 with 1).

**🔮 Predict:** how many rows does the consensus theorem need, and how many would a five-variable identity need?

In [2]:
AND = lambda x, y: x & y
OR = lambda x, y: x | y

theorems = {
    "T1 null element":   (lambda a: a | 1,               lambda a: 1,                 lambda a: a & 0,           lambda a: 0, 1),
    "T2 idempotence":    (lambda a: a | a,               lambda a: a,                 lambda a: a & a,           lambda a: a, 1),
    "T5 absorption":     (lambda a, b: a | (a & b),      lambda a, b: a,              lambda a, b: a & (a | b),  lambda a, b: a, 2),
    "T6 combining":      (lambda a, b: (a & b) | (a & NOT(b)), lambda a, b: a,        lambda a, b: (a | b) & (a | NOT(b)), lambda a, b: a, 2),
    "T7 redundant lit.": (lambda a, b: a | (NOT(a) & b), lambda a, b: a | b,          lambda a, b: a & (NOT(a) | b), lambda a, b: a & b, 2),
    "T8 consensus":      (lambda a, b, c: (a & b) | (NOT(a) & c) | (b & c), lambda a, b, c: (a & b) | (NOT(a) & c),
                          lambda a, b, c: (a | b) & (NOT(a) | c) & (b | c), lambda a, b, c: (a | b) & (NOT(a) | c), 3),
    "T9 De Morgan":      (lambda a, b: NOT(a | b),       lambda a, b: NOT(a) & NOT(b), lambda a, b: NOT(a & b), lambda a, b: NOT(a) | NOT(b), 2),
    "A3 distributivity": (lambda a, b, c: a & (b | c),   lambda a, b, c: (a & b) | (a & c),
                          lambda a, b, c: a | (b & c),   lambda a, b, c: (a | b) & (a | c), 3),
}
for name, (l, r, ld, rd, n) in theorems.items():
    ok, okd = same(l, r, n), same(ld, rd, n)
    print(f"{name:20s} {2 ** n:2d} rows   theorem {'ok' if ok else 'FAILS'}   dual {'ok' if okd else 'FAILS'}")
    assert ok and okd

# a five-variable identity: De Morgan for five inputs
n5 = lambda a, b, c, d, e: NOT(a | b | c | d | e)
p5 = lambda a, b, c, d, e: NOT(a) & NOT(b) & NOT(c) & NOT(d) & NOT(e)
print("five-variable De Morgan:", same(n5, p5, 5), "on", len(rows(5)), "rows")
assert same(n5, p5, 5)

T1 null element       2 rows   theorem ok   dual ok
T2 idempotence        2 rows   theorem ok   dual ok
T5 absorption         4 rows   theorem ok   dual ok
T6 combining          4 rows   theorem ok   dual ok
T7 redundant lit.     4 rows   theorem ok   dual ok
T8 consensus          8 rows   theorem ok   dual ok
T9 De Morgan          4 rows   theorem ok   dual ok
A3 distributivity     8 rows   theorem ok   dual ok
five-variable De Morgan: True on 32 rows


And a non-theorem, to see a failure: "cancellation" $A + B = A + C \Rightarrow B = C$ is false. Search for a counterexample instead of trusting intuition.

In [3]:
counter = [(a, b, c) for a, b, c in rows(3) if (a | b) == (a | c) and b != c]
print("rows where A+B = A+C but B != C:", counter)
assert counter    # e.g. A = 1 makes both sides 1 whatever B and C are

rows where A+B = A+C but B != C: [(1, 0, 1), (1, 1, 0)]


## Part 2 · Canonical forms, lookup tables and a census

A truth table becomes an expression by ORing its minterms (canonical SOP) or ANDing its maxterms (canonical POS). The lookup-table integer is the same information in one number.

In [4]:
NAMES = "ABCD"


def literal(var, bit, kind):
    pos = NAMES[var]
    neg = NAMES[var] + "'"
    if kind == "min":
        return pos if bit else neg
    return neg if bit else pos       # a maxterm is 0 exactly on its row


def canonical(table, n):
    on = [i for i in range(2 ** n) if (table >> i) & 1]
    off = [i for i in range(2 ** n) if not (table >> i) & 1]
    sop = " + ".join("".join(literal(k, r[k], "min") for k in range(n)) for r in (rows(n)[i] for i in on))
    pos = "".join("(" + " + ".join(literal(k, r[k], "max") for k in range(n)) + ")" for r in (rows(n)[i] for i in off))
    return on, off, sop, pos


maj = lambda a, b, c: int(a + b + c >= 2)
t = lut(maj, 3)
on, off, sop, pos = canonical(t, 3)
print("majority: lookup-table integer", t, "=", hex(t))
print("  sum of minterms", on, ":", sop)
print("  product of maxterms", off, ":", pos)
assert t == 232

majority: lookup-table integer 232 = 0xe8
  sum of minterms [3, 5, 6, 7] : A'BC + AB'C + ABC' + ABC
  product of maxterms [0, 1, 2, 4] : (A + B + C)(A + B + C')(A + B' + C)(A' + B + C)


**🔮 Predict:** of the 256 functions of three inputs, how many are *symmetric* (unchanged by permuting the inputs), how many *self-dual* ($f(\overline{x}) = \overline{f(x)}$), and how many *monotone* (never falling when an input rises)?

In [5]:
def bit(t, r):
    return (t >> (4 * r[0] + 2 * r[1] + r[2])) & 1


symmetric = [t for t in range(256) if all(bit(t, r) == bit(t, p) for r in rows(3) for p in itertools.permutations(r))]
self_dual = [t for t in range(256) if all(bit(t, r) == 1 - bit(t, tuple(1 - x for x in r)) for r in rows(3))]
monotone = [t for t in range(256) if all(bit(t, r) <= bit(t, s) for r in rows(3) for s in rows(3)
                                          if all(x <= y for x, y in zip(r, s)))]
print("symmetric:", len(symmetric), " self-dual:", len(self_dual), " monotone:", len(monotone))
print("majority (232) is in all three:", 232 in symmetric, 232 in self_dual, 232 in monotone)
x3 = lut(lambda a, b, c: a ^ b ^ c, 3)
print(f"XOR3 is {x3}: symmetric {x3 in symmetric}, self-dual {x3 in self_dual}, monotone {x3 in monotone}")
assert (len(symmetric), len(self_dual), len(monotone)) == (16, 16, 20)

symmetric: 16  self-dual: 16  monotone: 20
majority (232) is in all three: True True True
XOR3 is 150: symmetric True, self-dual True, monotone False


## Part 3 · Universal gates

NAND-NAND realizes any SOP, NOR-NOR any POS. Below, circuits are lists of gates and are checked against their specification on every row. Then a brute-force search finds the smallest NAND circuit for XOR.

In [6]:
GATES = {"NAND": lambda *x: 1 - int(all(x)), "NOR": lambda *x: 1 - int(any(x)), "INV": lambda x: 1 - x}


def simulate(netlist, inputs, out, n):
    tbl = 0
    for i, r in enumerate(rows(n)):
        val = dict(zip(inputs, r))
        for name, g, ins in netlist:
            val[name] = GATES[g](*(val[x] for x in ins))
        tbl |= val[out] << i
    return tbl


nand_nand = [("n1", "NAND", ["A", "B"]), ("n2", "NAND", ["C", "D"]), ("Y", "NAND", ["n1", "n2"])]
nor_nor = [("p", "NOR", ["A", "B"]), ("q", "NOR", ["C", "D"]), ("Y", "NOR", ["p", "q"])]
spec_sop = lut(lambda a, b, c, d: (a & b) | (c & d), 4)
spec_pos = lut(lambda a, b, c, d: (a | b) & (c | d), 4)
print("NAND-NAND = AB + CD:", simulate(nand_nand, list("ABCD"), "Y", 4) == spec_sop)
print("NOR-NOR = (A+B)(C+D):", simulate(nor_nor, list("ABCD"), "Y", 4) == spec_pos)
assert simulate(nand_nand, list("ABCD"), "Y", 4) == spec_sop and simulate(nor_nor, list("ABCD"), "Y", 4) == spec_pos


def smallest(op, n_inputs, target, max_gates=5):
    # breadth-first search over straight-line programs of 2-input gates on truth-table integers
    mask = (1 << 2 ** n_inputs) - 1
    start = tuple(lut(lambda *r, k=k: r[k], n_inputs) for k in range(n_inputs))
    frontier = [start]
    for depth in range(1, max_gates + 1):
        nxt = set()
        for sigs in frontier:
            for i in range(len(sigs)):
                for j in range(i, len(sigs)):
                    g = op(sigs[i], sigs[j]) & mask
                    if g == target:
                        return depth
                    nxt.add(tuple(sorted(set(sigs) | {g})))
        frontier = list(nxt)


nand_op = lambda x, y: ~(x & y)
print("smallest NAND2 circuit for XOR:", smallest(nand_op, 2, lut(lambda a, b: a ^ b, 2)), "gates")
print("smallest NAND2 circuit for OR:", smallest(nand_op, 2, lut(lambda a, b: a | b, 2)), "gates")
print("smallest NAND2 circuit for a 2:1 mux:", smallest(nand_op, 3, lut(lambda s, d0, d1: d1 if s else d0, 3)), "gates")
assert smallest(nand_op, 2, lut(lambda a, b: a ^ b, 2)) == 4

NAND-NAND = AB + CD: True
NOR-NOR = (A+B)(C+D): True
smallest NAND2 circuit for XOR: 4 gates
smallest NAND2 circuit for OR: 3 gates
smallest NAND2 circuit for a 2:1 mux: 4 gates


## Part 4 · CMOS gates as switch networks

A pull-down network is written as nested series (`S`) and parallel (`P`) connections of nMOS switches. Its dual, with series and parallel exchanged, is the pMOS pull-up network. nMOS conducts on a 1, pMOS on a 0.

In [7]:
def conducts(net, val, kind):
    if net[0] == "T":
        return val[net[1]] == (1 if kind == "n" else 0)
    parts = [conducts(x, val, kind) for x in net[1]]
    return all(parts) if net[0] == "S" else any(parts)


def dual(net):
    return net if net[0] == "T" else ("P" if net[0] == "S" else "S", [dual(x) for x in net[1]])


def size(net):
    return 1 if net[0] == "T" else sum(size(x) for x in net[1])


def gate_table(pdn, names):
    pun, tbl = dual(pdn), 0
    for i, r in enumerate(rows(len(names))):
        val = dict(zip(names, r))
        down, up = conducts(pdn, val, "n"), conducts(pun, val, "p")
        assert down != up, f"both or neither network conducts at {r}"
        tbl |= int(up) << i
    return tbl, size(pdn) + size(pun)


T = lambda x: ("T", x)
designs = {
    "INV  Y = A'":          (T("A"), ["A"], lambda a: 1 - a),
    "NAND2 Y = (AB)'":      (("S", [T("A"), T("B")]), ["A", "B"], lambda a, b: 1 - (a & b)),
    "NOR2 Y = (A+B)'":      (("P", [T("A"), T("B")]), ["A", "B"], lambda a, b: 1 - (a | b)),
    "OAI21 Y = (A(B+C))'":  (("S", [T("A"), ("P", [T("B"), T("C")])]), ["A", "B", "C"], lambda a, b, c: 1 - (a & (b | c))),
    "AOI22 Y = (AB+CD)'":   (("P", [("S", [T("A"), T("B")]), ("S", [T("C"), T("D")])]), list("ABCD"),
                             lambda a, b, c, d: 1 - ((a & b) | (c & d))),
}
for name, (pdn, names, spec) in designs.items():
    tbl, ntr = gate_table(pdn, names)
    print(f"{name:22s} transistors {ntr:2d}   correct: {tbl == lut(spec, len(names))}")
    assert tbl == lut(spec, len(names))

INV  Y = A'            transistors  2   correct: True
NAND2 Y = (AB)'        transistors  4   correct: True
NOR2 Y = (A+B)'        transistors  4   correct: True
OAI21 Y = (A(B+C))'    transistors  6   correct: True
AOI22 Y = (AB+CD)'     transistors  8   correct: True


Now break one: give the NAND2 a pull-up network that is a *copy* of its pull-down structure (series) instead of the dual. The checker reports the rows where neither network conducts (a floating output).

In [8]:
bad_pun = ("S", [T("A"), T("B")])
for r in rows(2):
    val = dict(zip("AB", r))
    down, up = conducts(("S", [T("A"), T("B")]), val, "n"), conducts(bad_pun, val, "p")
    state = "drives 0" if down and not up else "drives 1" if up and not down else "FLOATS" if not (up or down) else "SHORT"
    print("AB =", "".join(map(str, r)), "->", state)

AB = 00 -> drives 1
AB = 01 -> FLOATS
AB = 10 -> FLOATS
AB = 11 -> drives 0


Rows 01 and 10 float: with the pMOS in series, the pull-up needs both inputs low. The same mistake in Verilog shows up as `z` in the testbench (handout, step 6).

## What to take away

1. With two values per variable, perfect induction is a complete proof, and a truth table is just an integer.
2. Canonical SOP and POS always exist, which is why {AND, OR, NOT}, and hence NAND alone, can build anything.
3. Exhaustive search is a powerful checker for small blocks, and hopeless for large ones (§3.5).
4. A static CMOS gate is a pair of dual switch networks; exactly one must conduct in every input state.